# 🚀 KaizenStat — Quick Start (Tabular Data)

**Run this in under 2 minutes. No setup required.**

This notebook demonstrates the full KaizenStat pipeline on the Titanic dataset:

| Step | Method | What it does |
|------|--------|--------------|
| 1 | `fit()` | Register dataset, auto-detect task type |
| 2 | `health()` | Data Health Score 0–100 |
| 3 | `validate()` | Leakage + drift + statistical checks |
| 4 | `fix(safe=True)` | Preview then auto-heal data issues |
| 5 | `train()` | Benchmark 5 models, train the best |
| 6 | `debug_model()` | Root-cause failure analysis |
| 7 | `improve()` | Ranked improvement suggestions |
| 8 | `report()` | Terminal summary + HTML export |

---
> **Dataset:** Titanic (loaded automatically — no Kaggle login needed)
>
> **Time:** ~2 minutes on Colab CPU

In [ ]:
!pip install kaizenstat -q
print("✅ KaizenStat installed")

## Step 1 — Load Dataset

We drop high-cardinality ID/text columns (`Name`, `Ticket`, `PassengerId`, `Cabin`) that carry no
predictive signal and would confuse the feature-encoding step.

In [ ]:
import pandas as pd
from kaizenstat import DataDoctor

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Drop ID / free-text columns — no predictive value
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head()

## Step 2 — Fit the DataDoctor

`fit()` registers the dataset and auto-detects task type (classification / regression) and mode (tabular / NLP).

We predict `Survived` — binary classification (0 = died, 1 = survived).

In [ ]:
doctor = DataDoctor()
doctor.fit(df, target="Survived")

## Step 3 — Data Health Score

Scans for missing values, duplicates, outliers, class imbalance, and constant columns.
Returns a 0–100 score. **Target: > 80 for production-ready data.**

In [ ]:
health_result = doctor.health()
print(f"\n📊 Data Health Score: {health_result.score} / 100")

## Step 4 — Validate

Deeper statistical checks: data leakage, feature drift (KS test), multicollinearity (VIF).

In [ ]:
validation_result = doctor.validate()

## Step 5 — Auto-Fix

`fix(safe=True)` imputes missing values, drops constant columns, removes duplicates.
Only low-risk, reversible fixes are applied.

In [ ]:
fixed_df = doctor.fix(safe=True)
print(f"\n✅ Fixed dataset shape: {fixed_df.shape}")
print(f"   Missing values remaining: {fixed_df.isnull().sum().sum()}")

## Step 6 — Train

Benchmarks multiple algorithms, picks the winner, trains with cross-validation.

In [ ]:
train_result = doctor.train(cv=5)

print(f"\n🏆 Best model: {train_result.model_name}")
print(f"   Test score:  {train_result.test_score:.4f}")
print(f"   Train score: {train_result.train_score:.4f}")

## Step 7 — Debug Model

Root-cause failure analysis: data vs model blame, feature importances, failure slices.

In [ ]:
debug_result = doctor.debug_model()

print(f"\n🔍 Train score: {debug_result.train_score:.4f}")
print(f"   Test score:  {debug_result.test_score:.4f}")
print(f"   Gap:         {debug_result.gap:.4f}")

## Step 8 — Improve

Prioritised suggestions based on all previous steps: HIGH / MEDIUM / LOW priority.

In [ ]:
improvement_report = doctor.improve()

## Step 9 — Report

Terminal summary + self-contained HTML report you can share with your team.

In [ ]:
report_path = doctor.report(output_path="titanic_report.html")
print(f"\n📄 Report saved to: {report_path}")

In [ ]:
from IPython.display import IFrame, display
display(IFrame(src='titanic_report.html', width='100%', height='600px'))

In [ ]:
confidence = doctor.pipeline_confidence()
print(f"\n🎯 Pipeline Confidence: {confidence} / 100")

---
## Summary

```python
doctor = DataDoctor()
doctor.fit(df, target="Survived")   # auto-detect task type
doctor.health()                      # data health score 0–100
doctor.validate()                    # leakage + drift checks
doctor.fix(safe=True)                # preview then auto-heal
doctor.train()                       # benchmark + train best model
doctor.debug_model()                 # root-cause failure analysis
doctor.improve()                     # ranked improvement suggestions
doctor.report()                      # terminal summary + HTML export
```

**Next:** Try the [Basic Demo](demo_basic.ipynb) or run on your own CSV.

---
*KaizenStat v0.5.1 · [GitHub](https://github.com/kaizenstat-python/KaizenStat) · MIT License*